In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline

In [2]:
IN_PATH = 'CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
r = df['r'].dropna()
r2 = (r ** 2)

## 1. GARCH(1,1)

We estimate the canonical GARCH(1,1) on the REMX daily log returns (in percent) with constant mean and Gaussian innovations:

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2, \qquad z_t \sim \mathcal{N}(0,1).$$

The likelihood is maximized numerically. We report, per parameter, the estimate, standard error, **t-ratio** ($\hat\theta/\widehat{\mathrm{SE}}$) and the two-sided p-value, together with the model-level **log-likelihood, AIC, BIC**. A parameter is judged significant at the 5% level when $|t| > 1.96$ (equivalently $p < 0.05$). Standard errors are robust (sandwich form), so the conclusions also hold as a Quasi-MLE under conditional non-normality.


In [3]:
%%capture cap_garch
from arch import arch_model
from scipy.stats import norm

# 1. Returns in percent (standard scaling for GARCH stability)
r_pct = (df['r'].dropna() * 100.0)

# 2. Specify and fit GARCH(1,1) with constant mean and Gaussian innovations
am = arch_model(r_pct, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
res = am.fit(disp='off', cov_type='robust')

print(res.summary())
print()

# 3. Coefficient-level diagnostics (estimate / std err / t-ratio / p-value)
CRIT_10PCT = norm.ppf(1 - 0.10 / 2)  # 1.645
CRIT_5PCT  = norm.ppf(1 - 0.05 / 2)  # 1.960
CRIT_1PCT  = norm.ppf(1 - 0.01 / 2)  # 2.576

tbl = pd.DataFrame({
    'estimate':   res.params,
    'std_err':    res.std_err,
    't_ratio':    res.tvalues,
    'p_value':    res.pvalues,
})
tbl['significant_10pct'] = tbl['t_ratio'].abs() > CRIT_10PCT
tbl['significant_5pct']  = tbl['t_ratio'].abs() > CRIT_5PCT
tbl['significant_1pct']  = tbl['t_ratio'].abs() > CRIT_1PCT

print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()

# 4. Model-level criteria
print(f'Log-likelihood : {res.loglikelihood: .4f}')
print(f'AIC            : {res.aic: .4f}')
print(f'BIC            : {res.bic: .4f}')
print()

# 5. Stationarity / persistence check on the variance equation
alpha_hat = res.params.get('alpha[1]', float('nan'))
beta_hat  = res.params.get('beta[1]',  float('nan'))
persist   = alpha_hat + beta_hat
print(f'alpha[1] + beta[1] = {persist:.4f}   '
      f'({"stationary (<1)" if persist < 1 else "non-stationary (>=1)"})')
print()

# 6. Plain-language verdict at the 1%, 5% and 10% levels
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<10}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')

### Estimation results — interpretation

**Coefficient significance**

- `mu` $= -0.0062$ (in % / day) $\approx -1.6\%$ annualized, but $t = -0.17$ ($p = 0.87$): **not significant**. REMX's daily mean return is statistically indistinguishable from zero. This is typical for daily ETF returns and is not a problem; if a more parsimonious specification is preferred, the mean can be dropped (`mean='Zero'`) with virtually no change in log-likelihood.
- `omega` $= 0.0861$ ($t = 2.85$, $p = 0.004$): **significant**. Constant component of the conditional variance.
- `alpha[1]` $= 0.0892$ ($t = 5.54$, $p \approx 0$): **strongly significant**. There is a clear ARCH effect — yesterday's squared shock feeds today's variance with a weight of about 9%.
- `beta[1]` $= 0.8974$ ($t = 49.07$, $p \approx 0$): **highly significant**. Yesterday's variance dominates today's with a weight of about 90%. This is typically the most robust coefficient in the model.

**Conclusion of the estimation.** All three parameters of the variance equation — the ones that actually characterize the GARCH model — are significant at the 5% level. This validates GARCH(1,1) on REMX: the conditional variance cannot be collapsed to a constant, nor can the ARCH term be dropped.

**Persistence and stationarity**

$$\hat\alpha + \hat\beta = 0.9866 < 1$$

The variance process is stationary, but persistence is very high. Quantitative implications:

- **Implied unconditional variance:** $\hat\omega / (1 - \hat\alpha - \hat\beta) = 0.0861 / 0.0134 \approx 6.43$ in $(\%)^2$, i.e. an unconditional volatility of about **2.54% daily** $\approx$ **40.3% annualized**. For comparison, the sample standard deviation from the EDA was 2.29% daily / 36.4% annualized; the GARCH unconditional volatility sits slightly above because it incorporates the heavy-volatility clusters (e.g. March 2020).
- **Half-life of a variance shock:** $\ln(0.5) / \ln(\hat\alpha + \hat\beta) \approx 51$ trading days ($\approx$ 2.5 months). Once REMX enters a high-volatility regime, half of the shock decays only after roughly half a quarter.

**Goodness of fit.** Log-likelihood $= -5984.43$, AIC $= 11976.85$, BIC $= 12000.55$. These figures are only meaningful in **comparison** with alternative specifications — for now they constitute the baseline. When the GJR-GARCH, EGARCH-X and the same GARCH(1,1) with Student-$t$ innovations are estimated, they will be compared against these values (lower AIC / BIC = better model).

**Robustness caveats.** Standard errors are computed with a `robust` (sandwich) covariance estimator, so the reported $t$-ratios remain valid as **Quasi-MLE** even if the standardized residuals are non-Gaussian. The excess kurtosis of 3.7 reported in the EDA already hints at heavier-than-normal tails, which is the natural motivation for the Student-$t$ specification in the next step. A persistence of $0.99$ is typical but high; IGARCH or regime-switching models could provide marginal improvements but are refinements, not requirements.

# GARCH-X (tone) - tone matters?

# GARCH-X (article growth) - volume growth of articles matters?

# GARCHAND (tone) - tone sign matters?

# GARCHAND (article growth) - volume growth of articles sign matters?

# GARCHND (tone, article growth) - volatility regime matters?

# Generate report

In [4]:
from pathlib import Path

REPORTS_DIR = Path("REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)
(REPORTS_DIR / "_capture_garch.txt").write_text(cap_garch.stdout)

print(cap_garch.stdout)
print(f"Persisted capture -> REPORTS/_capture_garch.txt")


                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      r   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -5982.26
Distribution:                  Normal   AIC:                           11972.5
Method:            Maximum Likelihood   BIC:                           11996.2
                                        No. Observations:                 2765
Date:                Mon, May 25 2026   Df Residuals:                     2764
Time:                        20:09:31   Df Model:                            1
                                  Mean Model                                  
                  coef    std err          t      P>|t|       95.0% Conf. Int.
------------------------------------------------------------------------------
mu         -6.2207e-03  3.705e-02     -0.168      0.